# Tutorial - WebSensors Flow - Text Classification with Supervised Learning

This notebook presents a complete supervised text classification pipeline using **WebSensors Flow**.

The tutorial uses the **DMOZ Health** dataset, which contains textual descriptions of web pages related to health topics. Each record has a text and an associated category. The goal is to train a machine learning model capable of assigning health-related texts to their correct categories.

This is a typical text classification problem. Raw textual data must be ingested, cleaned, transformed into numerical representations, used to train candidate models, evaluated, and finally serialized for future use.

The pipeline is organized as a single end-to-end flow with four main steps:

1. data ingestion from the DMOZ Health CSV file;
2. text preprocessing and representation configuration;
3. pattern extraction, including train/test split, TF-IDF vectorization, `GridSearchCV`, and Multinomial Naive Bayes;
4. final model training and model serialization.


**WebSensors Flow** is designed as a lightweight and fully customizable observability framework for high-scale data and machine learning pipelines. Instead of imposing a rigid execution model, it allows each project to define its own steps, metrics, parameters, artifacts, and monitoring strategy.

In this example, each pipeline step reports relevant information such as dataset size, class distribution, preprocessing configuration, candidate model performance, selected hyperparameters, final evaluation metrics, and generated artifacts.

Only **MLflow** is enabled in this tutorial. Graylog and the FastAPI runner are not used here. 

The notebook defines the step classes directly in the cells and executes them one by one before running the complete flow. This makes the execution easier to inspect, debug, and understand before moving to a more automated or production-oriented scenario.

## 1. Imports and workspace

This notebook assumes that `websensors_flow` is already installed in the Python environment. The remaining libraries are standard data science libraries used by the example.

The workspace directory stores small artifacts generated by the steps, such as dataset samples, GridSearch results, classification reports, and the final serialized model bundle.


In [17]:
from __future__ import annotations

import json
import os
import platform
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd

from IPython.display import Markdown, display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline as SklearnPipeline

from websensors_flow import (
    MetricRecord,
    PipelineContext,
    PipelineStep,
    StepResult,
    build_pipeline_from_settings,
)
from websensors_flow.config import FlowSettings

WORK_DIR = Path("websensors_flow_notebook_workspace").resolve()
ARTIFACT_DIR = WORK_DIR / "artifacts"
REPORT_DIR = WORK_DIR / "reports"
MLRUNS_DIR = WORK_DIR / "mlruns"

for path in [WORK_DIR, ARTIFACT_DIR, REPORT_DIR, MLRUNS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DATASET_URL = "https://raw.githubusercontent.com/rmarcacini/text-collections/refs/heads/master/complete_texts_csvs/Dmoz-Health.csv"
RANDOM_STATE = 42

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Workspace: {WORK_DIR}")


Python: 3.13.5
Platform: Linux-6.12.88+deb13-rt-amd64-x86_64-with-glibc2.41
Workspace: /home/marcacini/Documents/PROJETOS/websensors-flow/notebooks/websensors_flow_notebook_workspace


## 2. Flow configuration in memory

In a regular project, this configuration could be stored in YAML. In this tutorial, it is kept as a Python dictionary so that the notebook is self-contained and easier to execute cell by cell.

The important points are:

- MLflow is enabled;
- Graylog is disabled;
- FastAPI is disabled;
- the MLflow tracking URI points to a local `mlruns` directory, so no external MLflow server is required;
- each step has a named configuration block that the step can access through `context.step_config`.


In [18]:
flow_config: dict[str, Any] = {
    "project": {
        "name": "websensors-flow-dmoz-health-notebook",
        "version": "1.0.0",
        "description": "Text classification tutorial using DMOZ Health and MLflow observability.",
    },
    "environment": {
        "name": "notebook",
        "owner": "tutorial",
        "tags": {
            "interface": "jupyter",
            "example": "dmoz-health",
        },
    },
    "runtime": {
        "report_dir": str(REPORT_DIR),
        "fail_fast": True,
        "include_traceback": True,
        "raise_on_failure": False,
        "console": {
            "enabled": True,
            "progress": True,
            "show_metrics": True,
        },
    },
    "observability": {
        "mlflow": {
            "enabled": True,
            "tracking_uri": f"http://127.0.0.1:5000",
            "experiment_name": "websensors-flow-dmoz-health-notebook",
            "run_name": "dmoz-health-notebook-flow",
            "http_request_timeout": 5,
            "connect_timeout_seconds": 3.0,
        },
        "graylog": {
            "enabled": False,
        },
    },
    "pipeline": {
        "params": {
            "dataset_url": DATASET_URL,
            "random_state": RANDOM_STATE,
            "task": "text_classification",
            "label_column": "class",
            "text_column": "text",
        },
    },
    "api": {
        "enabled": False,
    },
    "steps": [
        {
            "name": "ingest_data",
            "config": {
                "dataset_url": DATASET_URL,
                "max_rows": 2500,
                "min_text_length": 20,
                "sample_artifact_rows": 25,
                "artifact_dir": str(ARTIFACT_DIR / "01_ingest_data"),
            },
        },
        {
            "name": "preprocess_text",
            "config": {
                "text_column": "text",
                "label_column": "class",
                "min_class_count": 5,
                "lowercase": True,
                "tfidf": {
                    "stop_words": "english",
                    "strip_accents": "unicode",
                    "sublinear_tf": True,
                },
                "artifact_dir": str(ARTIFACT_DIR / "02_preprocess_text"),
            },
        },
        {
            "name": "extract_patterns",
            "config": {
                "test_size": 0.25,
                "random_state": RANDOM_STATE,
                "scoring": "f1_macro",
                "cv": 3,
                "grid": {
                    "tfidf__max_features": [2000, 5000],
                    "tfidf__ngram_range": [[1, 1], [1, 2]],
                    "classifier__alpha": [0.1, 0.5, 1.0],
                },
                "artifact_dir": str(ARTIFACT_DIR / "03_extract_patterns"),
            },
        },
        {
            "name": "postprocess_model",
            "config": {
                "registered_model_name": "websensors_flow_dmoz_health_nb",
                "artifact_dir": str(ARTIFACT_DIR / "04_postprocess_model"),
            },
        },
    ],
}

settings = FlowSettings.model_validate(flow_config).resolve_external_values()
settings


FlowSettings(source_path=None, project=ProjectConfig(name='websensors-flow-dmoz-health-notebook', version='1.0.0', description='Text classification tutorial using DMOZ Health and MLflow observability.'), environment=EnvironmentConfig(name='notebook', deployment_id=None, owner='tutorial', tags={'interface': 'jupyter', 'example': 'dmoz-health'}), runtime=RuntimeConfig(report_dir='/home/marcacini/Documents/PROJETOS/websensors-flow/notebooks/websensors_flow_notebook_workspace/reports', fail_fast=True, include_traceback=True, raise_on_failure=False, console=ConsoleConfig(enabled=True, progress=True, show_metrics=True)), observability=ObservabilityConfig(mlflow=MLflowConfig(enabled=True, tracking_uri='http://127.0.0.1:5000', tracking_uri_env='MLFLOW_TRACKING_URI', experiment_name='websensors-flow-dmoz-health-notebook', experiment_name_env='MLFLOW_EXPERIMENT_NAME', run_name='dmoz-health-notebook-flow', artifact_location=None, http_request_timeout=5, http_request_timeout_env='MLFLOW_HTTP_REQUE

The configuration above could be written as YAML in a real project. The structure would be the same: one main flow configuration, one block for observability, and one configuration block for each step.

In this notebook, the classes read the same values from `context.step_config`, but no YAML file is created.


## 3. Data objects passed between steps

WebSensors Flow does not require a fixed input/output object type. Each step can return any Python object as long as the step itself returns a `StepResult`.

For clarity, this notebook uses small dataclasses to describe the objects passed from one step to the next.


In [19]:
@dataclass
class IngestedDataset:
    """Raw dataset loaded from the external CSV source."""

    dataframe: pd.DataFrame
    source_url: str
    raw_dataset_path: Path
    sample_path: Path


@dataclass
class PreparedTextDataset:
    """Clean dataset and text representation settings prepared for model selection."""

    dataframe: pd.DataFrame
    text_column: str
    label_column: str
    vectorizer_params: dict[str, Any]
    prepared_dataset_path: Path
    class_distribution_path: Path


@dataclass
class ModelSelectionResult:
    """Result produced by GridSearchCV and the held-out test evaluation."""

    prepared: PreparedTextDataset
    best_estimator: SklearnPipeline
    best_params: dict[str, Any]
    best_cv_score: float
    test_metrics: dict[str, float]
    classification_report_dict: dict[str, Any]
    X_train: pd.Series
    X_test: pd.Series
    y_train: pd.Series
    y_test: pd.Series
    grid_results_path: Path
    classification_report_path: Path


@dataclass
class FinalModelBundle:
    """Final model trained on all available prepared data."""

    model: SklearnPipeline
    model_path: Path
    metadata_path: Path
    registered_model_name: str
    metadata: dict[str, Any]


## 4. Helper functions

These helper functions keep the step classes shorter. They are ordinary Python functions and do not depend on WebSensors Flow.


In [20]:
def ensure_dir(path: str | Path) -> Path:
    """Create a directory and return it as a Path object."""

    resolved = Path(path).resolve()
    resolved.mkdir(parents=True, exist_ok=True)
    return resolved


def save_json(data: Any, path: str | Path) -> Path:
    """Save a Python object as pretty JSON."""

    output_path = Path(path).resolve()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(data, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    return output_path


def save_dataframe(dataframe: pd.DataFrame, path: str | Path) -> Path:
    """Save a DataFrame as CSV with parent directory creation."""

    output_path = Path(path).resolve()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(output_path, index=False)
    return output_path


def show_step_result(title: str, result: StepResult) -> None:
    """Display a compact summary of one StepResult inside the notebook."""

    display(Markdown(f"### {title}"))
    display(Markdown(result.text))
    if result.metrics:
        display(pd.DataFrame([result.metrics]).T.rename(columns={0: "value"}))
    if result.params:
        display(pd.DataFrame([result.params]).T.rename(columns={0: "value"}))


## 5. Step 1: data ingestion

The ingestion step reads the external CSV file, validates the expected columns, optionally limits the number of rows for a fast tutorial run, and saves two artifacts:

- the raw dataset used by the tutorial;
- a small dataset sample used by MLflow to populate the dataset input information.

The step returns a `MetricRecord` named `dataset`. The MLflow observer uses this record to call `mlflow.log_input`, which helps populate the dataset information in the MLflow run.


In [21]:
class IngestDataStep(PipelineStep):
    """Load the DMOZ Health CSV file and expose dataset observability metadata."""

    name = "ingest_data"

    def execute(self, input: Any, context: PipelineContext) -> StepResult:
        config = context.step_config
        artifact_dir = ensure_dir(config.get("artifact_dir", ARTIFACT_DIR / self.name))

        dataset_url = config["dataset_url"]
        max_rows = config.get("max_rows")
        min_text_length = int(config.get("min_text_length", 1))
        sample_artifact_rows = int(config.get("sample_artifact_rows", 25))

        context.log.info(
            "Starting dataset ingestion.",
            params={"dataset_url": dataset_url, "max_rows": max_rows or 0},
        )

        try:
            dataframe = pd.read_csv(dataset_url)
        except Exception as exc:
            context.log.warning(
                "The remote dataset could not be loaded. A small local fallback dataset will be used.",
                metadata={"error_type": type(exc).__name__, "error_message": str(exc)},
            )
            dataframe = pd.DataFrame(
                {
                    "file_name": [
                        "health_001", "health_002", "health_003", "health_004", "health_005", "health_006",
                        "fitness_001", "fitness_002", "fitness_003", "fitness_004", "fitness_005", "fitness_006",
                        "nutrition_001", "nutrition_002", "nutrition_003", "nutrition_004", "nutrition_005", "nutrition_006",
                    ],
                    "text": [
                        "Medical information about symptoms treatment and patient care in hospitals.",
                        "Healthcare professionals discuss disease prevention and clinical diagnosis.",
                        "A hospital guide explains treatment options and medical appointments.",
                        "Patient education material about chronic disease and health services.",
                        "Clinical resources describe physicians medicine and public health.",
                        "Health article about diagnosis prevention and patient monitoring.",
                        "Fitness training plans improve strength flexibility and exercise routines.",
                        "Workout programs include aerobic exercise running and muscle training.",
                        "A guide to physical activity fitness goals and healthy movement.",
                        "Exercise routines help endurance strength and sports conditioning.",
                        "Training advice for gym workouts stretching and physical performance.",
                        "Fitness article about running training and active lifestyle habits.",
                        "Nutrition resources explain vitamins minerals diets and healthy meals.",
                        "A diet guide covers food calories protein and balanced nutrition.",
                        "Healthy eating article about fruits vegetables and meal planning.",
                        "Nutrition facts describe dietary choices weight control and food groups.",
                        "Dietary recommendations include protein fiber vitamins and hydration.",
                        "Food and nutrition article about meals calories and healthy ingredients.",
                    ],
                    "class": ["medical"] * 6 + ["fitness"] * 6 + ["nutrition"] * 6,
                }
            )

        expected_columns = {"file_name", "text", "class"}
        missing_columns = expected_columns.difference(dataframe.columns)
        if missing_columns:
            raise ValueError(f"The dataset is missing required columns: {sorted(missing_columns)}")

        dataframe = dataframe.dropna(subset=["text", "class"]).copy()
        dataframe["text"] = dataframe["text"].astype(str)
        dataframe["class"] = dataframe["class"].astype(str)
        dataframe = dataframe[dataframe["text"].str.len() >= min_text_length].reset_index(drop=True)

        if max_rows:
            dataframe = dataframe.head(int(max_rows)).copy()

        raw_dataset_path = save_dataframe(dataframe, artifact_dir / "raw_dataset.csv")
        sample_path = save_dataframe(dataframe.head(sample_artifact_rows), artifact_dir / "dataset_sample.csv")

        class_counts = dataframe["class"].value_counts()
        context.log.info(
            "Dataset ingestion finished.",
            metrics={
                "rows": len(dataframe),
                "classes": class_counts.shape[0],
                "min_class_count": int(class_counts.min()) if not class_counts.empty else 0,
                "max_class_count": int(class_counts.max()) if not class_counts.empty else 0,
            },
            metadata={"columns": list(dataframe.columns)},
        )

        output = IngestedDataset(
            dataframe=dataframe,
            source_url=dataset_url,
            raw_dataset_path=raw_dataset_path,
            sample_path=sample_path,
        )

        dataset_record = MetricRecord(
            name="dataset",
            params={
                "dataset_name": "Dmoz-Health",
                "source": dataset_url,
                "rows": len(dataframe),
                "text_column": "text",
                "label_column": "class",
            },
            metrics={
                "dataset_rows": len(dataframe),
                "dataset_classes": class_counts.shape[0],
            },
            metadata={
                "context": "training",
                "format": "csv",
            },
            artifacts={
                "dataset_sample": str(sample_path),
                "raw_dataset": str(raw_dataset_path),
            },
        )

        return StepResult(
            output=output,
            has_output=True,
            text=f"Loaded {len(dataframe)} documents from the DMOZ Health dataset.",
            metrics={
                "rows": len(dataframe),
                "classes": class_counts.shape[0],
                "min_class_count": int(class_counts.min()) if not class_counts.empty else 0,
                "max_class_count": int(class_counts.max()) if not class_counts.empty else 0,
            },
            params={
                "dataset_url": dataset_url,
                "max_rows": max_rows or "all",
                "min_text_length": min_text_length,
            },
            metadata={
                "artifact_dir": str(artifact_dir),
                "columns": list(dataframe.columns),
            },
            artifacts={
                "raw_dataset": str(raw_dataset_path),
                "dataset_sample": str(sample_path),
            },
            metric_records=[dataset_record],
        )


### Run the ingestion step directly

This direct execution is useful for teaching and debugging. The step receives the current `PipelineContext`, reads only its own configuration from `context.step_config`, and returns a `StepResult`.


In [22]:
manual_context = PipelineContext(
    settings=settings,
    run_id="notebook-step-by-step",
    pipeline_name=settings.project.name,
)

ingest_step = IngestDataStep()
manual_context.set_current_step(ingest_step.step_name, 1)
ingest_result = ingest_step.run(None, manual_context)
show_step_result("Ingestion result", ingest_result)

ingested = ingest_result.output
ingested.dataframe.head()


### Ingestion result

Loaded 2500 documents from the DMOZ Health dataset.

,value
rows,2500
classes,5
min_class_count,500
max_class_count,500


,value
dataset_url,https://raw.githubusercontent.com/rmarcacini/t...
max_rows,2500
min_text_length,20


,file_name,text,class
0,1578510.txt,Illinois Church Action on Alcohol and Addictio...,Addictions
1,1577747.txt,AA Statewide Meeting lists for all of Vermont....,Addictions
2,1578166.txt,Gracer Medical Group Dr. Richard Gracer is a p...,Addictions
3,1577381.txt,"Phoenix Meetings, events, and visitor informat...",Addictions
4,1578793.txt,American River Area Narcotics Anonymous Resour...,Addictions


## 6. Step 2: text preprocessing

This step prepares the dataset for text classification. It removes rare classes, normalizes the text column, and defines the TF-IDF representation parameters.

The vectorizer is not fitted here. It is configured here and fitted later inside `GridSearchCV`. This avoids data leakage because the TF-IDF vocabulary is learned only from each training fold during model selection.


In [23]:
class PreprocessTextStep(PipelineStep):
    """Prepare text and label columns for supervised text classification."""

    name = "preprocess_text"

    def execute(self, input: IngestedDataset, context: PipelineContext) -> StepResult:
        if not isinstance(input, IngestedDataset):
            raise TypeError("PreprocessTextStep expects an IngestedDataset object.")

        config = context.step_config
        artifact_dir = ensure_dir(config.get("artifact_dir", ARTIFACT_DIR / self.name))
        text_column = config.get("text_column", "text")
        label_column = config.get("label_column", "class")
        min_class_count = int(config.get("min_class_count", 2))
        lowercase = bool(config.get("lowercase", True))
        tfidf_config = dict(config.get("tfidf", {}))

        dataframe = input.dataframe[[text_column, label_column]].dropna().copy()
        dataframe[text_column] = dataframe[text_column].astype(str)
        dataframe[label_column] = dataframe[label_column].astype(str)

        if lowercase:
            dataframe[text_column] = dataframe[text_column].str.lower()

        before_rows = len(dataframe)
        class_counts = dataframe[label_column].value_counts()
        valid_classes = class_counts[class_counts >= min_class_count].index
        dataframe = dataframe[dataframe[label_column].isin(valid_classes)].reset_index(drop=True)
        after_rows = len(dataframe)

        if dataframe[label_column].nunique() < 2:
            raise ValueError("At least two classes are required after preprocessing.")

        prepared_dataset_path = save_dataframe(dataframe, artifact_dir / "prepared_dataset.csv")
        class_distribution = dataframe[label_column].value_counts().rename_axis("class").reset_index(name="count")
        class_distribution_path = save_dataframe(class_distribution, artifact_dir / "class_distribution.csv")

        vectorizer_params = {
            "stop_words": tfidf_config.get("stop_words", "english"),
            "strip_accents": tfidf_config.get("strip_accents", "unicode"),
            "sublinear_tf": bool(tfidf_config.get("sublinear_tf", True)),
        }

        context.log.info(
            "Text preprocessing finished.",
            metrics={
                "rows_before_filter": before_rows,
                "rows_after_filter": after_rows,
                "removed_rows": before_rows - after_rows,
                "classes_after_filter": dataframe[label_column].nunique(),
            },
            params=vectorizer_params,
        )

        output = PreparedTextDataset(
            dataframe=dataframe,
            text_column=text_column,
            label_column=label_column,
            vectorizer_params=vectorizer_params,
            prepared_dataset_path=prepared_dataset_path,
            class_distribution_path=class_distribution_path,
        )

        return StepResult(
            output=output,
            has_output=True,
            text=(
                f"Prepared {after_rows} documents across "
                f"{dataframe[label_column].nunique()} classes using TF-IDF settings."
            ),
            metrics={
                "rows_before_filter": before_rows,
                "rows_after_filter": after_rows,
                "removed_rows": before_rows - after_rows,
                "classes_after_filter": dataframe[label_column].nunique(),
            },
            params={
                "text_column": text_column,
                "label_column": label_column,
                "min_class_count": min_class_count,
                **vectorizer_params,
            },
            metadata={
                "artifact_dir": str(artifact_dir),
            },
            artifacts={
                "prepared_dataset": str(prepared_dataset_path),
                "class_distribution": str(class_distribution_path),
            },
        )


### Run the preprocessing step directly

The input of this step is the object returned by the ingestion step. The output is another object containing the cleaned data and the TF-IDF configuration.


In [24]:
preprocess_step = PreprocessTextStep()
manual_context.set_current_step(preprocess_step.step_name, 2)
preprocess_result = preprocess_step.run(ingested, manual_context)
show_step_result("Preprocessing result", preprocess_result)

prepared = preprocess_result.output
prepared.dataframe.head()


### Preprocessing result

Prepared 2500 documents across 5 classes using TF-IDF settings.

,value
rows_before_filter,2500
rows_after_filter,2500
removed_rows,0
classes_after_filter,5


,value
text_column,text
label_column,class
min_class_count,5
stop_words,english
strip_accents,unicode
sublinear_tf,True


,text,class
0,illinois church action on alcohol and addictio...,Addictions
1,aa statewide meeting lists for all of vermont....,Addictions
2,gracer medical group dr. richard gracer is a p...,Addictions
3,"phoenix meetings, events, and visitor informat...",Addictions
4,american river area narcotics anonymous resour...,Addictions


## 7. Step 3: pattern extraction with GridSearchCV

This step trains several TF-IDF + Multinomial Naive Bayes candidates and selects the best one by cross-validation.

The step also evaluates the selected model on a held-out test set. It returns several `MetricRecord` objects:

- one record for each GridSearch candidate;
- one record for each class in the test classification report.

The MLflow observer publishes these records as nested runs and artifacts, which makes the experiment easier to inspect.


In [25]:
class ExtractPatternsStep(PipelineStep):
    """Select a TF-IDF + MultinomialNB model using cross-validation."""

    name = "extract_patterns"

    def execute(self, input: PreparedTextDataset, context: PipelineContext) -> StepResult:
        if not isinstance(input, PreparedTextDataset):
            raise TypeError("ExtractPatternsStep expects a PreparedTextDataset object.")

        config = context.step_config
        artifact_dir = ensure_dir(config.get("artifact_dir", ARTIFACT_DIR / self.name))
        test_size = float(config.get("test_size", 0.25))
        random_state = int(config.get("random_state", RANDOM_STATE))
        scoring = config.get("scoring", "f1_macro")
        cv = int(config.get("cv", 3))
        raw_grid = dict(config.get("grid", {}))

        # YAML stores tuples as lists. Scikit-learn expects ngram_range values as tuples.
        param_grid = dict(raw_grid)
        if "tfidf__ngram_range" in param_grid:
            param_grid["tfidf__ngram_range"] = [tuple(value) for value in param_grid["tfidf__ngram_range"]]

        dataframe = input.dataframe
        X = dataframe[input.text_column]
        y = dataframe[input.label_column]

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=random_state,
            stratify=y,
        )

        model = SklearnPipeline(
            steps=[
                ("tfidf", TfidfVectorizer(**input.vectorizer_params)),
                ("classifier", MultinomialNB()),
            ]
        )

        context.log.info(
            "Starting GridSearchCV for text classification.",
            metrics={"train_rows": len(X_train), "test_rows": len(X_test)},
            params={"scoring": scoring, "cv": cv, "candidates": int(np.prod([len(v) for v in param_grid.values()]))},
        )

        search = GridSearchCV(
            estimator=model,
            param_grid=param_grid,
            scoring=scoring,
            cv=cv,
            n_jobs=-1,
            return_train_score=True,
        )
        search.fit(X_train, y_train)

        best_estimator = search.best_estimator_
        predictions = best_estimator.predict(X_test)

        test_accuracy = accuracy_score(y_test, predictions)
        test_f1_macro = f1_score(y_test, predictions, average="macro")
        test_f1_weighted = f1_score(y_test, predictions, average="weighted")
        report_dict = classification_report(y_test, predictions, output_dict=True, zero_division=0)

        grid_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
        grid_results_path = save_dataframe(grid_results, artifact_dir / "grid_search_results.csv")
        classification_report_path = save_json(report_dict, artifact_dir / "classification_report.json")

        candidate_records: list[MetricRecord] = []
        for row_index, row in grid_results.iterrows():
            params = dict(row["params"])
            candidate_records.append(
                MetricRecord(
                    name="model_candidate",
                    params={
                        "model_name": "tfidf_multinomial_nb",
                        "candidate_rank": int(row["rank_test_score"]),
                        "candidate_index": int(row_index),
                        **{key: str(value) for key, value in params.items()},
                    },
                    metrics={
                        "mean_test_score": float(row["mean_test_score"]),
                        "std_test_score": float(row["std_test_score"]),
                        "mean_train_score": float(row.get("mean_train_score", np.nan)),
                    },
                    metadata={
                        "scoring": scoring,
                        "cv": cv,
                    },
                )
            )

        class_records: list[MetricRecord] = []
        for class_name, values in report_dict.items():
            if not isinstance(values, dict) or class_name in {"accuracy", "macro avg", "weighted avg"}:
                continue
            class_records.append(
                MetricRecord(
                    name="class_metrics",
                    params={
                        "class_name": class_name,
                    },
                    metrics={
                        "precision": float(values.get("precision", 0.0)),
                        "recall": float(values.get("recall", 0.0)),
                        "f1_score": float(values.get("f1-score", 0.0)),
                        "support": float(values.get("support", 0.0)),
                    },
                    metadata={
                        "evaluation_split": "test",
                    },
                )
            )

        context.log.info(
            "GridSearchCV finished.",
            metrics={
                "best_cv_score": float(search.best_score_),
                "test_accuracy": float(test_accuracy),
                "test_f1_macro": float(test_f1_macro),
                "test_f1_weighted": float(test_f1_weighted),
            },
            params={key: str(value) for key, value in search.best_params_.items()},
        )

        output = ModelSelectionResult(
            prepared=input,
            best_estimator=best_estimator,
            best_params=dict(search.best_params_),
            best_cv_score=float(search.best_score_),
            test_metrics={
                "test_accuracy": float(test_accuracy),
                "test_f1_macro": float(test_f1_macro),
                "test_f1_weighted": float(test_f1_weighted),
            },
            classification_report_dict=report_dict,
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            grid_results_path=grid_results_path,
            classification_report_path=classification_report_path,
        )

        return StepResult(
            output=output,
            has_output=True,
            text=(
                "Selected the best TF-IDF + MultinomialNB candidate using GridSearchCV. "
                f"Best CV {scoring}: {search.best_score_:.4f}. "
                f"Test macro F1: {test_f1_macro:.4f}."
            ),
            metrics={
                "train_rows": len(X_train),
                "test_rows": len(X_test),
                "grid_candidates": len(grid_results),
                "best_cv_score": float(search.best_score_),
                "test_accuracy": float(test_accuracy),
                "test_f1_macro": float(test_f1_macro),
                "test_f1_weighted": float(test_f1_weighted),
            },
            params={
                "test_size": test_size,
                "random_state": random_state,
                "scoring": scoring,
                "cv": cv,
                **{key: str(value) for key, value in search.best_params_.items()},
            },
            metadata={
                "artifact_dir": str(artifact_dir),
                "model_family": "naive_bayes",
                "representation": "tfidf",
            },
            artifacts={
                "grid_search_results": str(grid_results_path),
                "classification_report": str(classification_report_path),
            },
            metric_records=candidate_records + class_records,
        )


### Run the model-selection step directly

This cell trains the candidate models. On the full DMOZ file, the number of rows is limited by `max_rows` in the ingestion configuration so that the tutorial remains reasonably fast.


In [26]:
extract_step = ExtractPatternsStep()
manual_context.set_current_step(extract_step.step_name, 3)
extract_result = extract_step.run(prepared, manual_context)
show_step_result("Pattern extraction result", extract_result)

selection = extract_result.output
pd.read_csv(selection.grid_results_path).head()


### Pattern extraction result

Selected the best TF-IDF + MultinomialNB candidate using GridSearchCV. Best CV f1_macro: 0.8562. Test macro F1: 0.8629.

,value
train_rows,1875.000000
test_rows,625.000000
grid_candidates,12.000000
best_cv_score,0.856217
test_accuracy,0.862400
test_f1_macro,0.862922
test_f1_weighted,0.862922


,value
test_size,0.25
random_state,42
scoring,f1_macro
cv,3
classifier__alpha,0.5
tfidf__max_features,5000
tfidf__ngram_range,"(1, 2)"


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier__alpha,param_tfidf__max_features,param_tfidf__ngram_range,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,mean_train_score,std_train_score
0,0.113189,0.010823,0.035613,0.007845,0.5,5000,"(1, 2)","{'classifier__alpha': 0.5, 'tfidf__max_feature...",0.847498,0.856717,0.864437,0.856217,0.006924,1,0.982391,0.984001,0.988816,0.985069,0.002730
1,0.097136,0.015964,0.019298,0.002255,1.0,5000,"(1, 2)","{'classifier__alpha': 1.0, 'tfidf__max_feature...",0.835421,0.858147,0.861680,0.851749,0.011636,2,0.975227,0.980000,0.980026,0.978418,0.002256
2,0.047097,0.006152,0.024511,0.000282,0.5,5000,"(1, 1)","{'classifier__alpha': 0.5, 'tfidf__max_feature...",0.839347,0.849655,0.865550,0.851517,0.010778,3,0.988786,0.988811,0.990409,0.989335,0.000759
3,0.174350,0.051882,0.041256,0.010890,0.1,5000,"(1, 2)","{'classifier__alpha': 0.1, 'tfidf__max_feature...",0.850773,0.844140,0.856326,0.850413,0.004982,4,0.994395,0.994397,0.996000,0.994931,0.000756
4,0.051932,0.000719,0.022397,0.001526,1.0,2000,"(1, 1)","{'classifier__alpha': 1.0, 'tfidf__max_feature...",0.840975,0.853285,0.850709,0.848323,0.005302,5,0.956082,0.966494,0.967186,0.963254,0.005079


## 8. Step 4: final model training and serialization

The model-selection step evaluates candidates on a train/test split. The final step uses the best parameters found by GridSearchCV and trains a final pipeline on all prepared data.

The serialized object is a dictionary containing:

- the final scikit-learn pipeline;
- the best parameters;
- the previous evaluation metrics;
- the text and label column names;
- the dataset source metadata.

The `StepResult` exposes the model artifact using the key `model_bundle`. The MLflow observer uses this key to log the scikit-learn model in MLflow, which helps populate the model information in the MLflow run.


In [27]:
class PostprocessModelStep(PipelineStep):
    """Train the final model on all prepared rows and serialize the model bundle."""

    name = "postprocess_model"

    def execute(self, input: ModelSelectionResult, context: PipelineContext) -> StepResult:
        if not isinstance(input, ModelSelectionResult):
            raise TypeError("PostprocessModelStep expects a ModelSelectionResult object.")

        config = context.step_config
        artifact_dir = ensure_dir(config.get("artifact_dir", ARTIFACT_DIR / self.name))
        registered_model_name = config.get("registered_model_name", settings.project.name)

        prepared = input.prepared
        dataframe = prepared.dataframe
        X = dataframe[prepared.text_column]
        y = dataframe[prepared.label_column]

        final_model = SklearnPipeline(
            steps=[
                ("tfidf", TfidfVectorizer(**prepared.vectorizer_params)),
                ("classifier", MultinomialNB()),
            ]
        )
        final_model.set_params(**input.best_params)
        final_model.fit(X, y)

        training_predictions = final_model.predict(X)
        training_accuracy = accuracy_score(y, training_predictions)
        training_f1_macro = f1_score(y, training_predictions, average="macro")

        metadata = {
            "registered_model_name": registered_model_name,
            "model_family": "MultinomialNB",
            "representation": "TF-IDF",
            "rows_used_for_final_training": len(dataframe),
            "classes": sorted(y.unique().tolist()),
            "best_params": {key: str(value) for key, value in input.best_params.items()},
            "best_cv_score": input.best_cv_score,
            "held_out_test_metrics": input.test_metrics,
            "final_training_metrics": {
                "training_accuracy": float(training_accuracy),
                "training_f1_macro": float(training_f1_macro),
            },
        }

        model_path = artifact_dir / "final_model_bundle.joblib"
        metadata_path = artifact_dir / "final_model_metadata.json"

        model_bundle = {
            "model": final_model,
            "metadata": metadata,
            "text_column": prepared.text_column,
            "label_column": prepared.label_column,
            "vectorizer_params": prepared.vectorizer_params,
        }
        joblib.dump(model_bundle, model_path)
        save_json(metadata, metadata_path)

        context.log.info(
            "Final model was trained and serialized.",
            metrics={
                "training_rows": len(dataframe),
                "training_accuracy": float(training_accuracy),
                "training_f1_macro": float(training_f1_macro),
            },
            params={"registered_model_name": registered_model_name, **metadata["best_params"]},
            artifacts={"model_bundle": str(model_path), "model_metadata": str(metadata_path)},
        )

        output = FinalModelBundle(
            model=final_model,
            model_path=model_path,
            metadata_path=metadata_path,
            registered_model_name=registered_model_name,
            metadata=metadata,
        )

        return StepResult(
            output=output,
            has_output=True,
            text=f"Trained and serialized the final model bundle at {model_path}.",
            metrics={
                "training_rows": len(dataframe),
                "training_classes": y.nunique(),
                "training_accuracy": float(training_accuracy),
                "training_f1_macro": float(training_f1_macro),
                "previous_best_cv_score": float(input.best_cv_score),
                **input.test_metrics,
            },
            params={
                "registered_model_name": registered_model_name,
                **metadata["best_params"],
            },
            metadata={
                "artifact_dir": str(artifact_dir),
                "registered_model_name": registered_model_name,
                "model_family": "MultinomialNB",
                "representation": "TF-IDF",
            },
            artifacts={
                "model_bundle": str(model_path),
                "model_metadata": str(metadata_path),
            },
            metric_records=[
                MetricRecord(
                    name="final_model",
                    params={
                        "model_name": registered_model_name,
                        "model_family": "MultinomialNB",
                        "representation": "TF-IDF",
                    },
                    metrics={
                        "training_accuracy": float(training_accuracy),
                        "training_f1_macro": float(training_f1_macro),
                        "test_accuracy": float(input.test_metrics["test_accuracy"]),
                        "test_f1_macro": float(input.test_metrics["test_f1_macro"]),
                    },
                    metadata={
                        "artifact": str(model_path),
                    },
                    artifacts={
                        "model_bundle": str(model_path),
                        "model_metadata": str(metadata_path),
                    },
                )
            ],
        )


### Run the final training step directly

After this cell, the model bundle is available as a local `joblib` artifact. The complete flow execution later will publish the same model to MLflow.


In [28]:
postprocess_step = PostprocessModelStep()
manual_context.set_current_step(postprocess_step.step_name, 4)
postprocess_result = postprocess_step.run(selection, manual_context)
show_step_result("Postprocessing result", postprocess_result)

final_bundle = postprocess_result.output
final_bundle.model_path


### Postprocessing result

Trained and serialized the final model bundle at /home/marcacini/Documents/PROJETOS/websensors-flow/notebooks/websensors_flow_notebook_workspace/artifacts/04_postprocess_model/final_model_bundle.joblib.

,value
training_rows,2500.000000
training_classes,5.000000
training_accuracy,0.968400
training_f1_macro,0.968409
previous_best_cv_score,0.856217
test_accuracy,0.862400
test_f1_macro,0.862922
test_f1_weighted,0.862922


,value
registered_model_name,websensors_flow_dmoz_health_nb
classifier__alpha,0.5
tfidf__max_features,5000
tfidf__ngram_range,"(1, 2)"


PosixPath('/home/marcacini/Documents/PROJETOS/websensors-flow/notebooks/websensors_flow_notebook_workspace/artifacts/04_postprocess_model/final_model_bundle.joblib')

## 9. Test the serialized model

The serialized bundle contains the complete preprocessing and classification pipeline. New texts can be classified by loading the `joblib` file and calling `predict` directly.


In [29]:
loaded_bundle = joblib.load(final_bundle.model_path)
loaded_model = loaded_bundle["model"]

examples = pd.Series(
    [
        "Information about hospital treatment and medical diagnosis for patients.",
        "Workout routines and physical exercise plans for better fitness.",
        "Healthy meals with vitamins, protein, vegetables, and balanced nutrition.",
    ]
)

predictions = loaded_model.predict(examples)
pd.DataFrame({"text": examples, "predicted_class": predictions})


,text,predicted_class
0,Information about hospital treatment and medic...,Medicine
1,Workout routines and physical exercise plans f...,Alternative
2,"Healthy meals with vitamins, protein, vegetabl...",Animal


## 10. Execute the complete flow in memory

The previous cells executed each step directly for didactic purposes. Now we instantiate the same classes and run the full flow with WebSensors Flow.

This execution activates the configured observers. Because MLflow is enabled, the run records:

- flow configuration;
- step parameters;
- step metrics;
- user logs emitted through `context.log`;
- dataset input information;
- GridSearch candidate metrics as nested runs;
- class-level metrics as nested runs;
- CSV and JSON artifacts;
- the final scikit-learn model.

No FastAPI server is used in this tutorial.


In [30]:
# MLflow system metrics are collected when the installed MLflow version supports them.
# The WebSensors Flow MLflow observer starts the run with log_system_metrics=True.
os.environ.setdefault("MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING", "true")

pipeline = build_pipeline_from_settings(settings)
pipeline.add(IngestDataStep())
pipeline.add(PreprocessTextStep())
pipeline.add(ExtractPatternsStep())
pipeline.add(PostprocessModelStep())

run_result = pipeline.run()

print(f"Pipeline status: {run_result.report.status}")
print(f"Run id: {run_result.report.run_id}")
print(f"MLflow tracking URI: {settings.observability.mlflow.tracking_uri}")
print(f"MLflow experiment: {settings.observability.mlflow.experiment_name}")


───────────────────────────────────────────────────────────────────────────────── WebSensors Flow ──────────────────────────────────────────────────────────────────────────────────

Resolved configuration
+-----------------------------------------------------------------------------------------------------------------------------+
| Item              | Value                                                                                                   |
|-------------------+---------------------------------------------------------------------------------------------------------|
| Config            | -                                                                                                       |
| Run ID            | b3134b15ae4e43258b81fa41e0c5ed5b                                                                        |
| Project           | websensors-flow-dmoz-health-notebook 1.0.0                                                              |
| Environment       | notebook                                                                                                |
| Reports           | /home/marcacini/Documents/PROJETOS/websensors-flow/notebooks/websensors_flow_notebook_workspace/reports |
| Graylog           | disabled tcp://-:-                                                                                      |
| MLflow            | enabled http://127.0.0.1:5000                                                                           |
| MLflow experiment | websensors-flow-dmoz-health-notebook                                                                    |
| API               | disabled 127.0.0.1:8000/runs                                                                            |
| Observers         | console, report, mlflow                                                                                 |
|-------------------+---------------------------------------------------------------------------------------------------------|
| Steps             | ingest_data -> preprocess_text -> extract_patterns -> postprocess_model                                 |
+-----------------------------------------------------------------------------------------------------------------------------+

──────────────────────────────────────────────────────────────────────────────────── Preflight ─────────────────────────────────────────────────────────────────────────────────────

+-------------------------------------- RUNNING preflight ---------------------------------------+
| Status        RUNNING                                                                          |
| Scope         preflight                                                                        |
| Name          console                                                                          |
| Message       Checking observer configuration.                                                 |
| phase         validate_ready                                                                   |
| target        terminal                                                                         |
| duration      -                                                                                |
+------------------------------------------------------------------------------------------------+

+----------------------------------------- OK preflight -----------------------------------------+
| Status        OK                                                                               |
| Scope         preflight                                                                        |
| Name          console                                                                          |
| Message       Observer configuration is ready.                                                 |
| phase         validate_ready                                                                   |
| target        terminal                                                                         |
| duration      0.003s                                                                           |
+------------------------------------------------------------------------------------------------+

...

Records emitted by postprocess_model
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Type        | Params                                                                           | Metrics                                                                         |
|-------------+----------------------------------------------------------------------------------+---------------------------------------------------------------------------------|
| final_model | model_name=websensors_flow_dmoz_health_nb; model_family=MultinomialNB;           | training_accuracy=0.9684; training_f1_macro=0.9684085465257924;                 |
|             | representation=TF-IDF                                                            | test_accuracy=0.8624; test_f1_macro=0.8629217079436676                          |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+

Progress  4/4 (100%)

+-------------------------------------------- OK pipeline ---------------------------------------------+
| Status              OK                                                                               |
| Scope               pipeline                                                                         |
| Name                websensors-flow-dmoz-health-notebook                                             |
| Message             Pipeline completed successfully.                                                 |
| duration            34.354s                                                                          |
| pipeline_success    1                                                                                |
| pipeline_failed     0                                                                                |
| steps_total         4                                                                                |
| steps_success       4                                                                                |
| steps_failed        0                                                                                |
| metrics_count       23                                                                               |
| params_count        20                                                                               |
| artifacts_count     8                                                                                |
| metric_records_count19                                                                               |
| warnings_count      0                                                                                |
+------------------------------------------------------------------------------------------------------+

Reports: /home/marcacini/Documents/PROJETOS/websensors-flow/notebooks/websensors_flow_notebook_workspace/reports

Pipeline status: success
Run id: b3134b15ae4e43258b81fa41e0c5ed5b
MLflow tracking URI: http://127.0.0.1:5000
MLflow experiment: websensors-flow-dmoz-health-notebook


## 11. Inspect the pipeline report

The report object is independent from MLflow. It is useful for programmatic checks after the run.


In [31]:
step_rows = []
for step in run_result.report.steps:
    step_rows.append(
        {
            "step": step.step_name,
            "status": step.status,
            "duration_seconds": round(step.duration_seconds or 0, 4),
            "metrics": step.metrics,
        }
    )

pd.DataFrame(step_rows)


,step,status,duration_seconds,metrics
0,ingest_data,success,0.8801,"{'rows': 2500, 'classes': 5, 'min_class_count'..."
1,preprocess_text,success,0.6484,"{'rows_before_filter': 2500, 'rows_after_filte..."
2,extract_patterns,success,1.7639,"{'train_rows': 1875, 'test_rows': 625, 'grid_c..."
3,postprocess_model,success,0.9003,"{'training_rows': 2500, 'training_classes': 5,..."


## 12. Open MLflow UI

The run was logged to a local MLflow tracking directory. From a terminal, run:

```bash
mlflow ui --backend-store-uri ./websensors_flow_notebook_workspace/mlruns
```

Then open the URL printed by MLflow, usually `http://127.0.0.1:5000`.

Inside MLflow, inspect the experiment named `websensors-flow-dmoz-health-notebook`. The run should contain metrics from each step, artifacts, dataset information, candidate model runs, class-level metric records, and the final model.


In [32]:
print("Command to open MLflow UI:")
print(f"mlflow ui --backend-store-uri {MLRUNS_DIR}")


Command to open MLflow UI:
mlflow ui --backend-store-uri /home/marcacini/Documents/PROJETOS/websensors-flow/notebooks/websensors_flow_notebook_workspace/mlruns
